# Classificação de Imagens com MobileNetV3 (timm)

Neste notebook, utilizaremos a biblioteca `timm` (PyTorch Image Models) para carregar um modelo **MobileNetV3 Small** pré-treinado e classificar as imagens contidas na pasta `imagens/`.

In [1]:
import os
import torch
import timm
from PIL import Image
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
import urllib.request
import json

# 1. Configuração do Dispositivo (GPU se disponível, caso contrário CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

# 2. Carregando o modelo timm/mobilenetv3_small_100.lamb_in1k
model_name = 'mobilenetv3_small_100.lamb_in1k'
model = timm.create_model(model_name, pretrained=True)
model = model.to(device)
model.eval()
print(f"Modelo {model_name} carregado com sucesso!")

Usando dispositivo: cpu


model.safetensors:   0%|          | 0.00/10.2M [00:00<?, ?B/s]

Modelo mobilenetv3_small_100.lamb_in1k carregado com sucesso!


c:\Users\Murilo\Documents\arquivos_programacao\NLW\projeto 2\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Murilo\.cache\huggingface\hub\models--timm--mobilenetv3_small_100.lamb_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [2]:
# 3. Preparando as transformações de imagem (ajuste para o que o modelo espera)
config = resolve_data_config({}, model=model)
transform = create_transform(**config)

# 4. Baixando as etiquetas (labels) do ImageNet se não existirem
labels_path = 'imagenet_labels.json'
if not os.path.exists(labels_path):
    url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
    urllib.request.urlretrieve(url, "imagenet_classes.txt")
    with open("imagenet_classes.txt", "r") as f:
        categories = [s.strip() for s in f.readlines()]
else:
    with open(labels_path, "r") as f:
        categories = json.load(f)

print("Transformações e etiquetas configuradas.")

Transformações e etiquetas configuradas.


In [3]:
# 5. Processando as imagens da pasta 'imagens'
pasta_imagens = 'imagens'
formatos_suportados = ('.png', '.jpg', '.jpeg', '.webp')
arquivos = [f for f in os.listdir(pasta_imagens) if f.lower().endswith(formatos_suportados)]

if not arquivos:
    print(f"Nenhum arquivo de imagem encontrado em {pasta_imagens}")
else:
    for arq in arquivos:
        caminho = os.path.join(pasta_imagens, arq)
        img = Image.open(caminho).convert('RGB')
        
        # Aplicar transformações e adicionar dimensão de batch
        tensor = transform(img).unsqueeze(0).to(device)
        
        with torch.no_grad():
            out = model(tensor)
        
        # Obter probabilidades (Softmax)
        probabilities = torch.nn.functional.softmax(out[0], dim=0)
        
        # Obter o top 1
        conf, index = torch.max(probabilities, 0)
        
        print(f"\nArquivo: {arq}")
        print(f"Predição: {categories[index.item()]} ({conf.item()*100:.2f}%)")


Arquivo: bird.jpeg
Predição: indigo bunting (11.57%)

Arquivo: kitchen.jpeg
Predição: plate rack (12.84%)

Arquivo: pizza.jpeg
Predição: pizza (60.77%)
